In [1]:
%pip install tensorflow scikit-learn pandas numpy matplotlib

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")

import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers as L
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

SEED = 42
tf.config.threading.set_intra_op_parallelism_threads(4)
tf.config.threading.set_inter_op_parallelism_threads(2)

FASHION_NAMES = ["T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
                 "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"]
CIFAR_NAMES = ["Airplane", "Automobile", "Bird", "Cat", "Deer", "Dog",
               "Frog", "Horse", "Ship", "Truck"]

In [3]:
def tf_dataset(x, y, training=False):
    ds = tf.data.Dataset.from_tensor_slices((x, y))
    if training:
        ds = ds.shuffle(len(y), seed=SEED, reshuffle_each_iteration=True)
    options = tf.data.Options()
    options.threading.private_threadpool_size = 3
    return ds.with_options(options).batch(128).prefetch(1)


def run_images(name, epochs, models=("DNN", "CNN")):
    keras.utils.set_random_seed(SEED)
    if name == "Fashion-MNIST":
        (x_dev, y_dev), (x_test, y_test) = keras.datasets.fashion_mnist.load_data()
        x_dev, x_test = x_dev[..., None], x_test[..., None]
        builder, labels = fashion_model, FASHION_NAMES
    else:
        (x_dev, y_dev), (x_test, y_test) = keras.datasets.cifar10.load_data()
        builder, labels = cifar_model, CIFAR_NAMES

    y_dev, y_test = y_dev.ravel(), y_test.ravel()
    x_dev, x_test = x_dev.astype("float32") / 255, x_test.astype("float32") / 255
    x_train, x_val, y_train, y_val = train_test_split(
        x_dev, y_dev, test_size=0.1, stratify=y_dev, random_state=SEED
    )
    val = tf_dataset(x_val, y_val)

    rows, details = [], {}
    for kind in models:
        keras.utils.set_random_seed(SEED)
        model = builder(kind)
        model.summary()
        model.compile(optimizer=keras.optimizers.Adam(1e-3),
                      loss="sparse_categorical_crossentropy", metrics=["accuracy"])
        history = model.fit(
            tf_dataset(x_train, y_train, training=True),
            validation_data=val, epochs=epochs, verbose=2, shuffle=False,
            callbacks=[
                keras.callbacks.ReduceLROnPlateau(
                    monitor="val_loss", factor=0.5, patience=3, min_lr=1e-5
                ),
                keras.callbacks.EarlyStopping(
                    monitor="val_loss", patience=6, restore_best_weights=True
                ),
            ],
        )
        probabilities = model.predict(tf_dataset(x_test, y_test), verbose=0)
        prediction = probabilities.argmax(axis=1)
        trainable = sum(int(np.prod(w.shape)) for w in model.trainable_weights)
        best = int(np.argmin(history.history["val_loss"]))
        rows.append({
            "model": kind, "best_epoch": best + 1,
            "val_accuracy": history.history["val_accuracy"][best],
            "test_accuracy": accuracy_score(y_test, prediction),
            "macro_f1": f1_score(y_test, prediction, average="macro"),
            "test_loss": float(keras.losses.sparse_categorical_crossentropy(
                y_test, probabilities
            ).numpy().mean()),
            "trainable": trainable, "non_trainable": model.count_params() - trainable,
            "total": model.count_params(),
        })
        matrix = confusion_matrix(y_test, prediction)
        report = classification_report(y_test, prediction, target_names=labels,
                                       digits=4, zero_division=0)
        print(report, flush=True)
        print(matrix, flush=True)
        details[kind] = {"history": history.history, "matrix": matrix.tolist(),
                         "report": report}
    results = pd.DataFrame(rows)
    print(results.to_string(index=False), flush=True)
    return results, details

In [4]:
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay


def show_image_results(results, details, classes):
    display(results.round(4))
    fig, axes = plt.subplots(1, 2, figsize=(15, 6), constrained_layout=True)
    for ax, (name, data) in zip(axes, details.items()):
        matrix = np.array(data["matrix"])
        print(f"\n{name}\n{data['report']}")
        ConfusionMatrixDisplay(matrix, display_labels=classes).plot(
            ax=ax, xticks_rotation=60, values_format="d", colorbar=False, cmap="Blues"
        )
        ax.set_title(name)
    plt.show()
    fig, ax = plt.subplots(figsize=(8, 4), constrained_layout=True)
    for name, data in details.items():
        losses = data["history"]["val_loss"]
        ax.plot(range(1, len(losses) + 1), losses, label=name)
    ax.set(xlabel="Epoch", ylabel="Validation cross-entropy")
    ax.legend()
    plt.show()

### 1 Fashion-MNIST 분류  
Macro F1은 10개 클래스의 F1을 각각 구한 뒤 평균한 값이다.

DNN은 이미지를 784개 값으로 펼친 뒤 폭 128과 64의 은닉층을 사용한다. CNN은 3×3 합성곱과 최대 풀링을 두 번 적용한 뒤 Flatten과 Dense를 거친다. 합성곱에는 same padding을 적용하므로 높이와 너비가 유지되며, 풀링을 거치면서 28 → 14 → 7이 된다. 두 모델 모두 기본 구조로 학습했다.

In [5]:
def fashion_model(kind):
    if kind == "DNN":
        hidden = [L.Flatten(), L.Dense(128, activation="relu"),
                  L.Dense(64, activation="relu")]
    else:
        hidden = [
            L.Conv2D(16, 3, padding="same", activation="relu"),
            L.MaxPooling2D(2),
            L.Conv2D(32, 3, padding="same", activation="relu"),
            L.MaxPooling2D(2), L.Flatten(), L.Dense(64, activation="relu"),
        ]
    return keras.Sequential([L.Input((28, 28, 1)), *hidden,
                             L.Dense(10, activation="softmax")], name=kind)

1) DNN 학습 및 시험 평가

In [6]:
fashion_dnn_results, fashion_dnn_details = run_images(
    "Fashion-MNIST", epochs=18, models=("DNN",)
)
display(fashion_dnn_results.round(4))

Model: "DNN"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 flatten (Flatten)           (None, 784)               0         
                                                                 
 dense (Dense)               (None, 128)               100480    
                                                                 
 dense_1 (Dense)             (None, 64)                8256      
                                                                 
 dense_2 (Dense)             (None, 10)                650       
                                                                 
Total params: 109,386
Trainable params: 109,386
Non-trainable params: 0
_________________________________________________________________
Epoch 1/18
422/422 - 1s - loss: 0.5585 - accuracy: 0.8039 - val_loss: 0.4200 - val_accuracy: 0.8523 - lr: 0.0010 - 531ms/epoch - 1ms/step
Epoch 2/18
422/422 - 0s - loss: 0.3975 - accuracy: 

,model,best_epoch,val_accuracy,test_accuracy,macro_f1,test_loss,trainable,non_trainable,total
0,DNN,16,0.9002,0.8924,0.8922,0.3203,109386,0,109386


시험 정확도는 89.24%, Macro F1은 0.8922이었다. 선택된 epoch는 16이었다.

2) CNN 학습 및 시험 평가

In [7]:
fashion_cnn_results, fashion_cnn_details = run_images(
    "Fashion-MNIST", epochs=18, models=("CNN",)
)
display(fashion_cnn_results.round(4))

Model: "CNN"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv2d (Conv2D)             (None, 28, 28, 16)        160       
                                                                 
 max_pooling2d (MaxPooling2D  (None, 14, 14, 16)       0         
 )                                                               
                                                                 
 conv2d_1 (Conv2D)           (None, 14, 14, 32)        4640      
                                                                 
 max_pooling2d_1 (MaxPooling  (None, 7, 7, 32)         0         
 2D)                                                             
                                                                 
 flatten_1 (Flatten)         (None, 1568)              0         
                                                                 
 dense_3 (Dense)             (None, 64)                100416  

,model,best_epoch,val_accuracy,test_accuracy,macro_f1,test_loss,trainable,non_trainable,total
0,CNN,14,0.9263,0.9169,0.9175,0.2515,105866,0,105866


시험 정확도는 91.69%, Macro F1은 0.9175였다. 선택된 epoch는 14였다.

3) DNN과 CNN 성능 비교

In [8]:
fashion_results = pd.concat(
    [fashion_dnn_results, fashion_cnn_results],
    ignore_index=True
)

display(fashion_results[["model", "test_accuracy", "macro_f1"]].round(4))

,model,test_accuracy,macro_f1
0,DNN,0.8924,0.8922
1,CNN,0.9169,0.9175


동일한 Fashion-MNIST 시험 데이터에서 CNN의 정확도는 91.69%로, DNN의 89.24%보다 2.45%p 높았다.  
클래스별 F1을 평균한 Macro F1도 CNN이 0.9175로 DNN의 0.8922보다 높았다. DNN은 이미지를 1차원 벡터로 펼쳐 입력하는 반면, CNN은 인접한 픽셀을 함께 처리해 옷의 윤곽과 같은 국소 특징을 학습할 수 있다. 이러한 구조적 차이가 이번 실험에서 CNN의 성능이 더 높게 나온 이유 중 하나로 볼 수 있다.

4) 총 모수 계산

In [9]:
display(fashion_results[["model", "trainable", "non_trainable", "total"]])

,model,trainable,non_trainable,total
0,DNN,109386,0,109386
1,CNN,105866,0,105866


DNN은 28X28 이미지를 784개 값으로 펼친 뒤, 출력 크기가 각각 128, 64, 10인 Dense 층에 입력한다. 따라서 총 모수는 다음과 같다.  
(784 + 1) X 128 + (128 + 1) X 64 + (64 + 1) X 10 = 100,480 + 8,256 + 650 = 109,386  

CNN은 3X3 합성곱층 두 개에 각각 16개와 32개의 필터를 사용한다. 합성곱에서 가로·세로 크기를 유지하고,  
MaxPooling을 두 번 거치며 크기가 28 -> 14 ->7로 줄어든다. 따라서 Flatten 이후 첫 Dense 층의 입력 수는 7X7X32 = 1,568이다. 총 모수는 다음과 같다.  
(3X3X1+1) X 16 + (3X3X16+1) X 32 + (1,568 + 1) X 64 + (64 + 1) X 10 = 105,866

### 2 CIFAR-10 분류

In [10]:
def dense_residual(x, units):
    shortcut = x
    x = L.Dense(units, use_bias=False)(x)
    x = L.BatchNormalization()(x)
    x = L.ReLU()(x)
    x = L.Dense(units, use_bias=False)(x)
    x = L.BatchNormalization()(x)
    return L.ReLU()(L.Add()([shortcut, x]))


def conv_residual(x, filters):
    shortcut = x
    x = L.Conv2D(filters, 3, padding="same", use_bias=False)(x)
    x = L.BatchNormalization()(x)
    x = L.ReLU()(x)
    x = L.Conv2D(filters, 3, padding="same", use_bias=False)(x)
    x = L.BatchNormalization()(x)
    return L.ReLU()(L.Add()([shortcut, x]))


def cifar_model(kind):
    inputs = keras.Input((32, 32, 3))
    if kind == "DNN":
        x = L.Flatten()(inputs)
        x = L.Dense(192, use_bias=False)(x)
        x = L.BatchNormalization()(x)
        x = L.ReLU()(x)
        x = dense_residual(x, 192)
        x = dense_residual(x, 192)
    else:
        x = L.Conv2D(32, 3, strides=2, padding="same", use_bias=False)(inputs)
        x = L.BatchNormalization()(x)
        x = L.ReLU()(x)
        x = conv_residual(x, 32)
        x = L.MaxPooling2D(2)(x)
        x = conv_residual(x, 32)
        x = L.GlobalAveragePooling2D()(x)
    return keras.Model(inputs, L.Dense(10, activation="softmax")(x),
                       name=f"Residual_{kind}")

1) DNN 학습 및 시험 평가

In [11]:
cifar_dnn_results, cifar_dnn_details = run_images(
    "CIFAR-10", epochs=22, models=("DNN",)
)
display(cifar_dnn_results.round(4))

Model: "Residual_DNN"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 input_3 (InputLayer)           [(None, 32, 32, 3)]  0           []                               
                                                                                                  
 flatten_2 (Flatten)            (None, 3072)         0           ['input_3[0][0]']                
                                                                                                  
 dense_5 (Dense)                (None, 192)          589824      ['flatten_2[0][0]']              
                                                                                                  
 batch_normalization (BatchNorm  (None, 192)         768         ['dense_5[0][0]']                
 alization)                                                                            

,model,best_epoch,val_accuracy,test_accuracy,macro_f1,test_loss,trainable,non_trainable,total
0,DNN,9,0.494,0.5003,0.4883,1.5803,741130,1920,743050


BN과 잔차 연결을 넣은 DNN의 시험 정확도는 50.03%, Macro F1은 0.4883이었다. 선택된 epoch는 9였다.

2) CNN 학습 및 시험 평가

In [12]:
cifar_cnn_results, cifar_cnn_details = run_images(
    "CIFAR-10", epochs=22, models=("CNN",)
)
display(cifar_cnn_results.round(4))

Model: "Residual_CNN"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 input_4 (InputLayer)           [(None, 32, 32, 3)]  0           []                               
                                                                                                  
 conv2d_2 (Conv2D)              (None, 16, 16, 32)   864         ['input_4[0][0]']                
                                                                                                  
 batch_normalization_5 (BatchNo  (None, 16, 16, 32)  128         ['conv2d_2[0][0]']               
 rmalization)                                                                                     
                                                                                                  
 re_lu_5 (ReLU)                 (None, 16, 16, 32)   0           ['batch_normalization_

,model,best_epoch,val_accuracy,test_accuracy,macro_f1,test_loss,trainable,non_trainable,total
0,CNN,19,0.737,0.7214,0.7169,0.8056,38378,320,38698


BN과 잔차 연결을 넣은 CNN의 시험 정확도는 72.14%, Macro F1은 0.7169였다. 선택된 epoch는 19이었다.

3) DNN과 CNN 성능 비교

In [13]:
cifar_results = pd.concat(
    [cifar_dnn_results, cifar_cnn_results],
    ignore_index=True
)

display(cifar_results[["model", "test_accuracy", "macro_f1"]].round(4))

,model,test_accuracy,macro_f1
0,DNN,0.5003,0.4883
1,CNN,0.7214,0.7169


동일한 CIFAR-10 시험 데이터에서 CNN의 정확도는 72.14%로, DNN의 50.03%보다 22.11%p 높았다.  
클래스별 F1을 평균한 Macro F1도 CNN이 0.7169로 DNN의 0.4883보다 높았다. 두 모델 모두 배치 정규화와 잔차 연결을 사용하였다. DNN은 이미지를 1차원 벡터로 펼쳐 입력하고, CNN은 인접한 픽셀을 함께 처리하여 윤곽이나 질감 같은 특징을 학습한다.  
이러한 구조적 차이가 이번 실험에서 CNN의 분류 성능이 더 높게 나타난 이유 중 하나로 볼 수 있다.

4)  총 모수 계산

In [14]:
display(cifar_results[["model", "trainable", "non_trainable", "total"]])

,model,trainable,non_trainable,total
0,DNN,741130,1920,743050
1,CNN,38378,320,38698


DNN은 32 x 32 x 3 이미지를 Flatten하여 3,072개의 입력값으로 변환한다. 첫 Dense 층은 192개의 뉴런을 사용하며, 두 잔차 블록에는 입력과 출력 크기가 모두 192인 Dense 층이 총 4개 있다. 이 5개의 Dense 층은 편향을 사용하지 않고, 마지막 Dense 층은 10개 클래스에 대한 출력을 계산하며 편향을 포함한다. Batch Normalization 층은 총 5개이며, 각 뉴런마다 학습 모수 2개와 비학습 모수 2개를 저장한다. Flatten과 잔차 연결의 덧셈에는 모수가 없다.
학습 모수 = (3,072 x 192) + 4 x (192 x 192) + (192 + 1) x 10 + 5 x (192 x 2) = 741,130
비학습 모수 = 5 x (192 x 2) = 1,920
총 모수 = 741,130 + 1,920 = 743,050

CNN의 첫 번째 3 x 3 합성곱층은 RGB 입력의 3개 채널에 32개의 필터를 적용한다. 두 잔차 블록 안에는 입력 채널과 출력 채널이 모두 32개인 3 x 3 합성곱층이 총 4개 있다. 합성곱층 5개는 편향을 사용하지 않는다. Global Average Pooling을 거치면 채널별 평균값 32개가 나오고, 마지막 Dense 층은 이 값들을 받아 편향을 포함한 10개 클래스 출력을 계산한다. Batch Normalization 층은 총 5개이며, 각 채널마다 학습 모수 2개와 비학습 모수 2개를 저장한다. 풀링과 잔차 연결의 덧셈에는 모수가 없다.
학습 모수 = (3 x 3 x 3) x 32 + 4 x (3 x 3 x 32) x 32 + (32 + 1) x 10 + 5 x (32 x 2) = 38,378
비학습 모수 = 5 x (32 x 2) = 320
총 모수 = 38,378 + 320 = 38,698

### 3 Bikeshare 시간당 대여 건수

In [15]:
from io import BytesIO
from math import comb
from pathlib import Path
from urllib.request import urlopen
from zipfile import ZipFile

from numpy.lib.stride_tricks import sliding_window_view
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

BIKE_URL = "https://archive.ics.uci.edu/static/public/275/bike+sharing+dataset.zip"
WINDOW = 20
GROUPS = {"mnth": [0, 1], "hr": [2, 3], "workingday": [4],
          "weathersit": [5, 6, 7, 8], "temp": [9]}


def load_bike(local_path=None):
    if local_path is None:
        with urlopen(BIKE_URL, timeout=60) as response:
            local_path = BytesIO(response.read())
    if isinstance(local_path, (str, Path)) and Path(local_path).suffix.lower() == ".csv":
        frame = pd.read_csv(local_path)
    else:
        with ZipFile(local_path) as archive:
            with archive.open("hour.csv") as csv:
                frame = pd.read_csv(csv)

    dates = pd.to_datetime(frame["dteday"]) + pd.to_timedelta(frame["hr"], unit="h")
    data = frame.assign(timestamp=dates).sort_values("timestamp").reset_index(drop=True)
    assert not data["timestamp"].duplicated().any()
    month = 2 * np.pi * (data["mnth"].to_numpy() - 1) / 12
    hour = 2 * np.pi * data["hr"].to_numpy() / 24
    weather = np.eye(4, dtype="float32")[data["weathersit"].to_numpy() - 1]
    features = np.column_stack([
        np.sin(month), np.cos(month), np.sin(hour), np.cos(hour),
        data["workingday"], weather, data["temp"],
    ]).astype("float32")
    # Target time t uses t-19,...,t covariates. No rental count is an input.
    windows = sliding_window_view(features, WINDOW, axis=0).transpose(0, 2, 1)
    date_windows = sliding_window_view(data["timestamp"].to_numpy(), WINDOW)
    continuous = (date_windows[:, -1] - date_windows[:, 0]) == np.timedelta64(WINDOW - 1, "h")
    return (np.ascontiguousarray(windows[continuous]),
            data["cnt"].to_numpy(dtype="float32")[WINDOW - 1:][continuous],
            data["timestamp"].to_numpy()[WINDOW - 1:][continuous])


def bike_model(kind, shape):
    if kind == "DNN":
        hidden = [L.Flatten(), L.Dense(128, activation="relu")] ## DNN
    else:
        hidden = [L.Conv1D(32, 3, activation="relu"), L.MaxPooling1D(2), L.Flatten()] ## 1D-CNN
    return keras.Sequential([L.Input(shape), *hidden, L.Dense(64, activation="relu"),
                             L.Dense(1)], name=kind.replace("-", "_"))

##RF
def forecast(data, model, kind, mean, scale):
    if kind == "RF":
        values = model.predict(data.reshape(len(data), -1))
    else:
        # Direct batches keep repeated PFI/SHAP predictions lightweight.
        values = np.concatenate([
            model(data[start:start + 1024], training=False).numpy().ravel()
            for start in range(0, len(data), 1024)
        ])
        values = mean + scale * values
    return np.maximum(values, 0)

In [16]:
def grouped_pfi(predict, x, y, repeats=12):
    rng = np.random.default_rng(SEED)
    permutations = [rng.permutation(len(x)) for _ in range(repeats)]
    baseline = mean_squared_error(y, predict(x))
    scores = []
    for name, channels in GROUPS.items():
        changes = []
        for order in permutations:
            changed = x.copy()
            changed[:, :, channels] = x[order][:, :, channels]
            changes.append(mean_squared_error(y, predict(changed)) - baseline)
        scores.append((name, np.mean(changes), np.std(changes, ddof=1)))
    return pd.DataFrame(scores, columns=["feature", "delta_mse", "permutation_sd"])


def grouped_shap(predict, explained, background):
    # Enumerate all 2**5 coalitions. Replace absent groups with whole
    # trajectories from the same background row and average predictions.
    p = len(GROUPS)
    values = np.empty((1 << p, len(explained)))
    for mask in range(1 << p):
        present = [channel for j, group in enumerate(GROUPS.values())
                   if mask & (1 << j) for channel in group]
        if mask == 0:
            values[mask] = predict(background).mean()
        elif mask == (1 << p) - 1:
            values[mask] = predict(explained)
        else:
            hybrid = np.broadcast_to(
                background[None], (len(explained), *background.shape)
            ).copy()
            hybrid[:, :, :, present] = explained[:, None, :, present]
            predictions = predict(hybrid.reshape(-1, *explained.shape[1:]))
            values[mask] = predictions.reshape(len(explained), len(background)).mean(axis=1)

    phi = np.zeros((len(explained), p))
    for j in range(p):
        for mask in range(1 << p):
            if mask & (1 << j):
                continue
            weight = 1 / (p * comb(p - 1, mask.bit_count()))
            phi[:, j] += weight * (values[mask | (1 << j)] - values[mask])
    np.testing.assert_allclose(values[0] + phi.sum(axis=1), values[-1], atol=1e-4)
    return phi, values[0, 0], values[-1]

In [17]:
def run_bike(local_path=None):
    x, y, dates = load_bike(local_path)
    train_end, val_end = int(0.70 * len(y)), int(0.85 * len(y))
    x_train, x_val, x_test = np.split(x, [train_end, val_end])
    y_train, y_val, y_test = np.split(y, [train_end, val_end])
    test_dates = dates[val_end:]
    for name, time in zip(("train", "validation", "test"),
                          np.split(dates, [train_end, val_end])):
        print(name, len(time), time[0], time[-1], flush=True)
    mean, scale = float(y_train.mean()), float(y_train.std())
    rng = np.random.default_rng(SEED)
    background = x_train[rng.choice(len(x_train), 32, replace=False)]
    example_ids = np.sort(rng.choice(len(x_test), 128, replace=False))
    explained = x_test[example_ids]
    # Explain a common sample chosen by its observed count.
    local = int(np.argmax(y_test[example_ids]))

    metrics, pfi, shap_summary, local_shap = [], [], [], []
    for kind in ("DNN", "1D-CNN", "RF"):
        keras.utils.set_random_seed(SEED)
        if kind == "RF":
            model = RandomForestRegressor(
                n_estimators=120, min_samples_leaf=3, max_features=0.6,
                random_state=SEED, n_jobs=3
            )
            model.fit(x_train.reshape(len(x_train), -1), y_train)
            best_epoch = None
        else:
            model = bike_model(kind, x.shape[1:])
            model.compile(optimizer=keras.optimizers.Adam(1e-3), loss="mse")
            history = model.fit(
                tf_dataset(x_train, (y_train - mean) / scale, training=True),
                validation_data=tf_dataset(x_val, (y_val - mean) / scale),
                epochs=70, verbose=0, shuffle=False,
                callbacks=[keras.callbacks.EarlyStopping(
                    monitor="val_loss", patience=10, restore_best_weights=True
                )],
            )
            best_epoch = int(np.argmin(history.history["val_loss"])) + 1

        def predict(data):
            return forecast(data, model, kind, mean, scale)

        pred = predict(x_test)
        result = dict(model=kind, best_epoch=best_epoch,
                      rmse=mean_squared_error(y_test, pred)**0.5,
                      mae=mean_absolute_error(y_test, pred), r2=r2_score(y_test, pred))
        print(result, flush=True)
        metrics.append(result)
        pfi.append(grouped_pfi(predict, x_test, y_test).assign(model=kind))
        phi, base, explained_predictions = grouped_shap(predict, explained, background)
        shap_summary.append(pd.DataFrame({
            "feature": list(GROUPS), "mean_abs_shap": np.abs(phi).mean(axis=0),
            "model": kind,
        }))
        local_shap.append(dict(model=kind, date=str(test_dates[example_ids[local]]),
                               actual=int(y_test[example_ids[local]]),
                               prediction=explained_predictions[local], baseline=base,
                               **dict(zip(GROUPS, phi[local]))))
    return (pd.DataFrame(metrics), pd.concat(pfi, ignore_index=True),
            pd.concat(shap_summary, ignore_index=True), pd.DataFrame(local_shap))

1) DNN·1D-CNN·RF 예측 성능 비교

In [18]:
#모델비교
bike_results, pfi_results, shap_results, local_results = run_bike()
display(bike_results.round(4))

train 11179 2011-01-01T19:00:00.000000000 2012-06-09T17:00:00.000000000
validation 2395 2012-06-09T18:00:00.000000000 2012-09-17T12:00:00.000000000
test 2396 2012-09-17T13:00:00.000000000 2012-12-31T23:00:00.000000000
{'model': 'DNN', 'best_epoch': 16, 'rmse': 127.74621218088033, 'mae': 90.76947021484375, 'r2': 0.6403458118438721}
{'model': '1D-CNN', 'best_epoch': 5, 'rmse': 119.71492831932031, 'mae': 80.61404418945312, 'r2': 0.684146523475647}
{'model': 'RF', 'best_epoch': None, 'rmse': 129.49880630235413, 'mae': 90.18679302462205, 'r2': 0.6304096387042619}


,model,best_epoch,rmse,mae,r2
0,DNN,16.0,127.7462,90.7695,0.6403
1,1D-CNN,5.0,119.7149,80.6140,0.6841
2,RF,NaN,129.4988,90.1868,0.6304


1D-CNN은 RMSE(119.71)와 MAE(80.61)가 가장 낮고 R^2(0.6841)가 가장 높아, 이번 시험 데이터에서 가장 좋은 예측 성능을 보였다. DNN과 RF를 비교하면 DNN의 RMSE가 더 낮았고, RF의 MAE가 더 낮았다.

2) PFI·SHAP 변수 중요도 비교

In [19]:
display(pfi_results.pivot(index="feature", columns="model", values="delta_mse").round(2))
display(shap_results.pivot(index="feature", columns="model", values="mean_abs_shap").round(2))

model,1D-CNN,DNN,RF
feature,,,
hr,44022.86,46495.50,42460.74
mnth,1769.91,1512.34,429.16
temp,2373.50,1935.61,9272.18
weathersit,2841.92,3356.92,1901.02
workingday,8684.12,9420.40,8717.05


model,1D-CNN,DNN,RF
feature,,,
hr,107.98,109.14,98.81
mnth,11.07,15.61,4.92
temp,17.66,17.03,25.36
weathersit,20.90,19.59,11.42
workingday,22.10,23.12,20.98


PFI는 변수 값을 섞었을 때 평균제곱오차(MSE)가 얼마나 증가하는지 나타내며, 평균 절대 SHAP 값은 각 변수가 개별 예측값을 움직인 크기를 평균한 것이다. 두 방법 모두 세 모델에서 시간대(hr)를 가장 중요한 변수로 평가하였다. 두 번째로 중요한 변수는 DNN과 1D-CNN에서는 근무일 여부(workingday), Random Forest에서는 기온(temp)이었다.

평균 절대 SHAP 값은 각 변수가 모델의 예측 대여 건수를 평균적으로 얼마나 크게 움직였는지 나타낸다. 세 모델 모두 시간대(hr)의 값이 가장 컸으며, DNN은 109.14건, 1D-CNN은 107.98건, Random Forest는 98.81건이었다. 두 번째로 중요한 변수는 DNN과 1D-CNN에서 근무일 여부(workingday), Random Forest에서는 기온(temp)이었다. 특히 기온의 중요도는 Random Forest에서 25.36건으로, DNN(17.03건)과 1D-CNN(17.66건)보다 높았다. 월(mnth)은 세 모델에서 모두 중요도가 가장 낮았다.